# Relaxed density matrix from CCSD calculation
**Author:** Egor Trushin  
**Date created:** 13/08/2025  
**Last modified:** 24/10/2026  

In [1]:
import pyscf
print("PySCF version:", pyscf.__version__)

from pyscf import scf, cc, gto

from utils.relaxed_dm_cc import relaxed_rdm1

PySCF version: 2.14.0


The relaxed CCSD density matrix is computed with `relaxed_rdm1` from [`utils/relaxed_dm_cc.py`](https://github.com/EgorTrushin/pyscf-notebooks/blob/main/notebooks/utils/relaxed_dm_cc.py), a copy of the module from [PyOEP](https://github.com/EgorTrushin/PyOEP). Besides the CCSD response density, it includes the orbital relaxation obtained from the Z-vector (CPHF) equation, so that the density reproduces analytic one-electron properties such as the dipole moment.

**Note:** the module also provides the relaxed CCSD(T) density, computed from the same converged CCSD object with `relaxed_rdm1(mf_cc, "ccsd(t)")`. Since CCSD(T) is the default, the method has to be given explicitly for CCSD.

In [2]:
ORBITAL_BASIS = "aug-cc-pVTZ"
GEOM = "C 0.000000    0.000000   -0.646514; O 0.000000    0.000000    0.484886"

mol = gto.M(atom=GEOM, basis=ORBITAL_BASIS)
mol.verbose = 0
mol.symmetry = False

mf = scf.RHF(mol)
mf.kernel()
print(f"Hartree-Fock total energy: {mf.e_tot:15.12f}")
dm_hf = mf.make_rdm1()

mf_cc = cc.CCSD(mf)
mf_cc.kernel()
print(f"CCSD correlation energy:     {mf_cc.e_corr:15.12f}", )
print(f"CCSD total energy:         {mf_cc.e_tot:15.12f}")
dm_ccsd_unrelaxed = mf_cc.make_rdm1(ao_repr=True)
dm_ccsd_relaxed = relaxed_rdm1(mf_cc, "ccsd")

Hartree-Fock total energy: -112.781038130830


CCSD correlation energy:     -0.392786437643
CCSD total energy:         -113.173824568473


In [3]:
print("Hartree-Fock dipole moment")
_ = mf.dip_moment(dm=dm_hf)
print("CCSD dipole moment (unrelaxed)")
_ = mf.dip_moment(dm=dm_ccsd_unrelaxed, unit='AU')
_ = mf.dip_moment(dm=dm_ccsd_unrelaxed)
print("CCSD dipole moment (relaxed)")
_ = mf.dip_moment(dm=dm_ccsd_relaxed, unit='AU')
_ = mf.dip_moment(dm=dm_ccsd_relaxed)

Hartree-Fock dipole moment
Dipole moment(X, Y, Z, Debye): -0.00000, -0.00000, -0.28269


CCSD dipole moment (unrelaxed)
Dipole moment(X, Y, Z, A.U.):  0.00000,  0.00000,  0.05219


Dipole moment(X, Y, Z, Debye):  0.00000,  0.00000,  0.13265


CCSD dipole moment (relaxed)
Dipole moment(X, Y, Z, A.U.): -0.00000, -0.00000,  0.02401


Dipole moment(X, Y, Z, Debye): -0.00000, -0.00000,  0.06104


**Results from Molpro**  
~~~ 
 Reference energy                   -112.781038130534
 CCSD correlation energy              -0.392786368232
!CCSD total energy                  -113.173824498766

 Dipole moments (unrelaxed):            0.00000000     0.00000000     0.05222402
 Orbital relaxation contribution:       0.00000000     0.00000000    -0.02818700
!CCSD STATE 1.1 Dipole moment           0.00000000     0.00000000     0.02403702
 Dipole moment /Debye                   0.00000000     0.00000000     0.06109601
~~~

The results were obtained with the following input:
~~~
basis={
default,aug-cc-pVTZ
}

symmetry,nosym

angstrom
geometry={
2

C        0.000000    0.000000   -0.646514
O        0.000000    0.000000    0.484886
}

hf,maxit=30

{CCSD;core;dm,1325.1}
~~~